# MER VIS galaxy cutouts on ESA Datalabs

Select the **euclid-tools** kernel. Run cells in order using Shift+Enter.
Upload this notebook, the companion Python file, and MER_VIS_datalabs_manifest.csv
into the same folder in your persistent workspace. No image download URLs are used.
The source mosaics are read from the mounted Euclid Q1 data volume.


In [ ]:
from pathlib import Path
import sys, shutil, csv
import numpy, astropy
print('Python:', sys.version)
print('Astropy:', astropy.__version__)
print('Notebook folder:', Path.cwd())
WORK = Path('/media/user/Euclid_MER_VIS')
# /media/user is the personal workspace in ESA's tutorial.
# If your instance uses another persistent workspace mount, change WORK.
WORK.mkdir(parents=True, exist_ok=True)
OUTPUT = WORK / 'cutouts'
MANIFEST = Path('MER_VIS_datalabs_manifest.csv').resolve()
assert MANIFEST.is_file(), 'Upload the manifest into the notebook folder'
assert Path('make_MER_VIS_datalabs_cutouts.py').is_file(), 'Upload the Python helper'
print('Output:', OUTPUT)
print('Filesystem free GiB:', round(shutil.disk_usage(WORK).free / 2**30, 2))
print('Also check your account quota in Datalabs; filesystem free is not your quota.')


## Check the source paths and estimated output size

The source paths were retrieved from Euclid TAP's datalabs_path metadata.
If paths are missing, check the mounted Q1 volume in the official Cutouts tutorial.
Do not download full mosaics to repair a missing mount.


In [ ]:
import make_MER_VIS_datalabs_cutouts as mer
report = mer.preflight(MANIFEST)
report


## Create a five galaxy pilot

This writes derived science cutouts to your workspace, not to the shared volume.
The native projection and WCS are retained; no deprojection or rotation is applied.


In [ ]:
mer.run(MANIFEST, OUTPUT, limit=5)


## Inspect the first cutout

Check the galaxy position, size and whether the requested sky box is clipped.
SCIENCE cutouts do not include RMS, weight, segmentation or quality maps.


In [ ]:
from astropy.io import fits
from astropy.visualization import simple_norm
import matplotlib.pyplot as plt
files = sorted(OUTPUT.glob('*_MER_VIS_cutout.fits'))
assert files, 'No cutouts created; inspect the processing report'
with fits.open(files[0], checksum=True) as hdul:
    data = hdul[0].data
    header = hdul[0].header
    print(files[0].name, data.shape, 'PARTIAL:', header['PARTIAL'])
    plt.figure(figsize=(7,7))
    plt.imshow(data, origin='lower', cmap='gray',
               norm=simple_norm(data, 'asinh', percent=99.5))
    plt.plot(header['CENX']-1, header['CENY']-1, 'r+')
    plt.title(header['OBJECT'])
    plt.show()


## Run all galaxies after checking the pilot and storage quota

This cell is deliberately commented out. Remove the leading # when ready.
Valid existing cutouts are checked and skipped on a rerun. Each completed
object is checkpointed in MER_VIS_processing_manifest.csv. A stopped kernel
can be restarted and the batch rerun. Keep the browser session open until complete.


In [ ]:
# mer.run(MANIFEST, OUTPUT, limit=None)


## Review the report

Errors and PARTIAL=True need inspection. Five successful pilot products do not
guarantee every remaining galaxy will succeed. Files persist only if OUTPUT is
in your persistent personal/team workspace. Save this notebook with Ctrl+S.


In [ ]:
from collections import Counter
with (OUTPUT/'MER_VIS_processing_manifest.csv').open(newline='') as f:
    results = list(csv.DictReader(f))
print(Counter(r['status'] for r in results))
print('Report:', OUTPUT/'MER_VIS_processing_manifest.csv')
